In [1]:
import time
import nest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import neo
import quantities as pq
import re
from elephant.statistics import isi, cv, mean_firing_rate
from elephant.conversion import BinnedSpikeTrain
from elephant.spike_train_correlation import corrcoef
from pathlib import Path

## import model implementation
import network
## import (default) parameters (network, simulation, stimulus)
from network_params import default_net_dict as net_dict
from sim_params import default_sim_dict as sim_dict
from stimulus_params import default_stim_dict as stim_dict

# Import library NeuroRing for FPGA and pyxrt
import neuroring
import pyxrt
from utils_binding import *   # provides .index and .bitstreamFile

# Create network and connect neurons
net = network.Network(sim_dict, net_dict, stim_dict)
net.create()
net.connect()

print(net.pops)


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.9.0
 Built: Oct  2 2025 06:57:01

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.

Data will be written to: /home/miahafiz/NeuroRing/host_py/data/
  Directory already existed. Old data will be overwritten.


RNG seed: 55
Total number of virtual processes: 4
Creating neuronal populations.

Mar 05 16:44:30 SimulationManager::set_status [Info]: 
    Temporal resolution changed from 0.1 to 0.1 ms.
Connecting neuronal populations recurrently.

Mar 05 16:45:16 NodeManager::prepare_nodes [Info]: 
    Preparing 77169 nodes for simulation.
[NodeCollection(metadata=None, model=iaf_psc_exp, size=20683, first=1, last=20683), NodeCollection(metadata=None, model=iaf_psc_exp, size=5834, first=20684, last=26517), NodeCollection(metadata=None, model=iaf_psc_ex

In [2]:
param_dict = {
    'dt': 0.1,
    'tau_m': 10.0,
    'tau_syn': 0.5,
    'C_m': 250.0,
    'E_L': -65.0,
    't_ref_steps': 20,
    'V_th_abs': -50.0,
    'V_reset_abs': -65.0,
}
# 1 for recording, 0 for not recording spike
record_status = 1

host = neuroring.NeuroRingHost(net, 4096, 7000, 20, 2, param_dict, record_status, "/home/miahafiz/NeuroRing/_build_dir.hw.NUM_4096.CORE_10.FREQ_300/krnl_neuroring_hw.xclbin")

Extracting synapse information...
Loading synapse data from syndata_total77169_NperCU4096_SperCU7000.npy
Loading synapse FPGA data from synfpga_total77169_NperCU4096_SperCU7000.npy

Distributing 77169 neurons across 20 compute units on 2 FPGAs:
Kernels per FPGA: [10, 10]

FPGA 0 (Kernels: 10):
  Kernel 0 (Global ID: 0): neurons 1 to 4096 (total: 4096)
  Kernel 1 (Global ID: 1): neurons 4097 to 8192 (total: 4096)
  Kernel 2 (Global ID: 2): neurons 8193 to 12288 (total: 4096)
  Kernel 3 (Global ID: 3): neurons 12289 to 16384 (total: 4096)
  Kernel 4 (Global ID: 4): neurons 16385 to 20480 (total: 4096)
  Kernel 5 (Global ID: 5): neurons 20481 to 24576 (total: 4096)
  Kernel 6 (Global ID: 6): neurons 24577 to 28672 (total: 4096)
  Kernel 7 (Global ID: 7): neurons 28673 to 32768 (total: 4096)
  Kernel 8 (Global ID: 8): neurons 32769 to 36864 (total: 4096)
  Kernel 9 (Global ID: 9): neurons 36865 to 40960 (total: 4096)

FPGA 1 (Kernels: 10):
  Kernel 0 (Global ID: 10): neurons 40961 to 45056

In [3]:
host.initialize_devices()
print("Initialized devices")

Initialized device 0 with XCLBIN UUID: 5bf43ddb-2f3a-6c4c-746a-07f4b2b07fe8
{'simulation_time': 1, 'amount_of_cores': 20, 'neuron_start': 1, 'neuron_total': 4096, 'device': None, 'xclbin': None, 'uuid': None, 'kernel_name': None, 'kernel': None, 'neuron_per_cu': 4096, 'synapse_total_per_cu': 7000, 'param_dict': {'dt': 0.1, 'tau_m': 10.0, 'tau_syn': 0.5, 'C_m': 250.0, 'E_L': -65.0, 't_ref_steps': 20, 'V_th_abs': -50.0, 'V_reset_abs': -65.0}, 'record_status': 1, 'synapseListHandle': None, 'header_words': 12800000, 'header_bytes': 51200000, 'tail_words_capacity': 57344000, 'tail_bytes_capacity': 229376000, 'bo_size': 280576000, 'core_id': 0}
Initialized kernel NeuroRing:{NeuroRing_0} and SynapseRouter:{SynapseRouter_0} on device <pyxrt.device object at 0x7f54310d1cf0>
Allocated BO of 280576000 bytes (header 51200000, tail 229376000)
{'simulation_time': 1, 'amount_of_cores': 20, 'neuron_start': 4097, 'neuron_total': 4096, 'device': None, 'xclbin': None, 'uuid': None, 'kernel_name': None, '

In [4]:
host.kernels_per_fpga[0][0].upload_synapse_list(host.synapse_fpga[0])
host.kernels_per_fpga[0][1].upload_synapse_list(host.synapse_fpga[1])
host.kernels_per_fpga[0][2].upload_synapse_list(host.synapse_fpga[2])
host.kernels_per_fpga[0][3].upload_synapse_list(host.synapse_fpga[3])
host.kernels_per_fpga[0][4].upload_synapse_list(host.synapse_fpga[4])
host.kernels_per_fpga[0][5].upload_synapse_list(host.synapse_fpga[5])
host.kernels_per_fpga[0][6].upload_synapse_list(host.synapse_fpga[6])
host.kernels_per_fpga[0][7].upload_synapse_list(host.synapse_fpga[7])
host.kernels_per_fpga[0][8].upload_synapse_list(host.synapse_fpga[8])
host.kernels_per_fpga[0][9].upload_synapse_list(host.synapse_fpga[9])

host.kernels_per_fpga[1][0].upload_synapse_list(host.synapse_fpga[10])
host.kernels_per_fpga[1][1].upload_synapse_list(host.synapse_fpga[11])
host.kernels_per_fpga[1][2].upload_synapse_list(host.synapse_fpga[12])
host.kernels_per_fpga[1][3].upload_synapse_list(host.synapse_fpga[13])
host.kernels_per_fpga[1][4].upload_synapse_list(host.synapse_fpga[14])
host.kernels_per_fpga[1][5].upload_synapse_list(host.synapse_fpga[15])
host.kernels_per_fpga[1][6].upload_synapse_list(host.synapse_fpga[16])
host.kernels_per_fpga[1][7].upload_synapse_list(host.synapse_fpga[17])
host.kernels_per_fpga[1][8].upload_synapse_list(host.synapse_fpga[18])
#host.kernels_per_fpga[1][9].upload_synapse_list(host.synapse_fpga[19])


Uploaded synapse list to <pyxrt.kernel object at 0x7f543162fe30>
Uploaded synapse list to <pyxrt.kernel object at 0x7f57b571fb30>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310e3670>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310e31f0>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310e0270>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310ea3f0>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310e9970>
Uploaded synapse list to <pyxrt.kernel object at 0x7f5431658ef0>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310ea7f0>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54336fe170>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310d98f0>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310ea9f0>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310eab30>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310eac70>
Uploaded synapse list to <pyxrt.kernel object at 0x7f54310eae70>
Uploaded synapse list to 

In [5]:
timestep = 100000
start_time = time.time()
host.kernels_per_fpga[0][0].run_neuroring(timestep)
host.kernels_per_fpga[0][0].run_synapserouter(timestep)
host.kernels_per_fpga[0][1].run_neuroring(timestep)
host.kernels_per_fpga[0][1].run_synapserouter(timestep)
host.kernels_per_fpga[0][2].run_neuroring(timestep)
host.kernels_per_fpga[0][2].run_synapserouter(timestep)
host.kernels_per_fpga[0][3].run_neuroring(timestep)
host.kernels_per_fpga[0][3].run_synapserouter(timestep)
host.kernels_per_fpga[0][4].run_neuroring(timestep)
host.kernels_per_fpga[0][4].run_synapserouter(timestep)
host.kernels_per_fpga[0][5].run_neuroring(timestep)
host.kernels_per_fpga[0][5].run_synapserouter(timestep)
host.kernels_per_fpga[0][6].run_neuroring(timestep)
host.kernels_per_fpga[0][6].run_synapserouter(timestep)
host.kernels_per_fpga[0][7].run_neuroring(timestep)
host.kernels_per_fpga[0][7].run_synapserouter(timestep)
host.kernels_per_fpga[0][8].run_neuroring(timestep)
host.kernels_per_fpga[0][8].run_synapserouter(timestep)
host.kernels_per_fpga[0][9].run_neuroring(timestep)
host.kernels_per_fpga[0][9].run_synapserouter(timestep)

host.kernels_per_fpga[1][0].run_neuroring(timestep)
host.kernels_per_fpga[1][0].run_synapserouter(timestep)
host.kernels_per_fpga[1][1].run_neuroring(timestep)
host.kernels_per_fpga[1][1].run_synapserouter(timestep)
host.kernels_per_fpga[1][2].run_neuroring(timestep)
host.kernels_per_fpga[1][2].run_synapserouter(timestep)
host.kernels_per_fpga[1][3].run_neuroring(timestep)
host.kernels_per_fpga[1][3].run_synapserouter(timestep)
host.kernels_per_fpga[1][4].run_neuroring(timestep)
host.kernels_per_fpga[1][4].run_synapserouter(timestep)
host.kernels_per_fpga[1][5].run_neuroring(timestep)
host.kernels_per_fpga[1][5].run_synapserouter(timestep)
host.kernels_per_fpga[1][6].run_neuroring(timestep)
host.kernels_per_fpga[1][6].run_synapserouter(timestep)
host.kernels_per_fpga[1][7].run_neuroring(timestep)
host.kernels_per_fpga[1][7].run_synapserouter(timestep)
host.kernels_per_fpga[1][8].run_neuroring(timestep)
host.kernels_per_fpga[1][8].run_synapserouter(timestep)
host.kernels_per_fpga[1][9].run_neuroring(timestep)
host.kernels_per_fpga[1][9].run_synapserouter(timestep)

host.kernels_per_fpga[0][0].wait_for_kernel()
host.kernels_per_fpga[0][1].wait_for_kernel()
host.kernels_per_fpga[0][2].wait_for_kernel()
host.kernels_per_fpga[0][3].wait_for_kernel()
host.kernels_per_fpga[0][4].wait_for_kernel()
host.kernels_per_fpga[0][5].wait_for_kernel()
host.kernels_per_fpga[0][6].wait_for_kernel()
host.kernels_per_fpga[0][7].wait_for_kernel()
host.kernels_per_fpga[0][8].wait_for_kernel()
host.kernels_per_fpga[0][9].wait_for_kernel()

host.kernels_per_fpga[1][0].wait_for_kernel()
host.kernels_per_fpga[1][1].wait_for_kernel()
host.kernels_per_fpga[1][2].wait_for_kernel()
host.kernels_per_fpga[1][3].wait_for_kernel()
host.kernels_per_fpga[1][4].wait_for_kernel()
host.kernels_per_fpga[1][5].wait_for_kernel()
host.kernels_per_fpga[1][6].wait_for_kernel()
host.kernels_per_fpga[1][7].wait_for_kernel()
host.kernels_per_fpga[1][8].wait_for_kernel()
host.kernels_per_fpga[1][9].wait_for_kernel()

end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")


Time taken: 8.334981918334961 seconds


In [6]:
host.get_spike_recorder_array(timestep)
print("Getting spike recorder array")
host.plot_spike_recorder_array(filename="spike_recorder.png", start_tick=0, end_tick=100)
print("Plotted spike recorder array")

# save host.spikeidx and host.neuronidx to csv
np.savetxt("spikeidx.csv", host.spikeidx, delimiter=",")
np.savetxt("neuronidx.csv", host.neuronidx, delimiter=",")
print("Saved spikeidx and neuronidx to csv")

Getting spike recorder array
Plotted spike recorder array
Saved spikeidx and neuronidx to csv


In [7]:
# Statistics analysis: pair spikes with neuron IDs and map to cortical layers
spikeidx = host.spikeidx
neuronidx = host.neuronidx

layer_labels = ["2/3E", "2/3I", "4E", "4I", "5E", "5I", "6E", "6I"]
layer_file = "data/population_nodeids.dat"
tick_to_ms = 0.1  # from sim_resolution = 0.1 ms

spike_ticks = np.asarray(spikeidx, dtype=np.int64)
neuron_ids = np.asarray(neuronidx, dtype=np.int64)

if spike_ticks.shape[0] != neuron_ids.shape[0]:
    raise ValueError(
        f"Length mismatch: spikeidx={spike_ticks.shape[0]}, neuronidx={neuron_ids.shape[0]}"
    )

# One row per spike event
spike_events = pd.DataFrame(
    {
        "spike_tick": spike_ticks,
        "neuron_id": neuron_ids,
    }
)
spike_events["spike_time_ms"] = spike_events["spike_tick"].astype(float) * tick_to_ms
spike_events = spike_events.sort_values(["neuron_id", "spike_time_ms"]).reset_index(drop=True)

# Load layer start/end neuron IDs (inclusive)
layer_table = pd.read_csv(
    layer_file,
    sep=r"\s+",
    header=None,
    names=["start_id", "end_id"],
)
if len(layer_table) != 8:
    raise ValueError(f"Expected 8 layer rows, got {len(layer_table)}")

layer_table["layer"] = layer_labels

starts = layer_table["start_id"].to_numpy(dtype=np.int64)
ends = layer_table["end_id"].to_numpy(dtype=np.int64)

layer_idx = np.full(neuron_ids.shape, -1, dtype=np.int64)
for i, (start_id, end_id) in enumerate(zip(starts, ends)):
    mask = (neuron_ids >= start_id) & (neuron_ids <= end_id)
    layer_idx[mask] = i

if np.any(layer_idx < 0):
    n_unmapped = int(np.sum(layer_idx < 0))
    raise ValueError(f"Found {n_unmapped} spikes with neuron IDs outside layer ranges")

spike_events["layer_idx"] = layer_idx
spike_events["layer"] = pd.Categorical(
    [layer_labels[i] for i in layer_idx],
    categories=layer_labels,
    ordered=True,
)

t_stop_ms = float(spike_events["spike_time_ms"].max() + tick_to_ms) if len(spike_events) else tick_to_ms

print(f"Total spikes paired: {len(spike_events):,}")
print(f"Simulation stop time for analysis: {t_stop_ms:.3f} ms")
print(layer_table)
spike_events.head()

Total spikes paired: 2,457,948
Simulation stop time for analysis: 10000.000 ms
   start_id  end_id layer
0         1   20683  2/3E
1     20684   26517  2/3I
2     26518   48432    4E
3     48433   53911    4I
4     53912   58761    5E
5     58762   59826    5I
6     59827   74221    6E
7     74222   77169    6I


,spike_tick,neuron_id,spike_time_ms,layer_idx,layer
0,7771,2,777.1,0,2/3E
1,23214,2,2321.4,0,2/3E
2,36159,2,3615.9,0,2/3E
3,38002,2,3800.2,0,2/3E
4,85836,2,8583.6,0,2/3E


In [8]:
# Load NEST spike recorder data and compute metrics with the same pipeline as NeuroRing

nest_data_dir = Path("data")
nest_pattern = re.compile(r"spike_recorder-(\d+)-(\d+)\.dat$")

all_nest_files = sorted(nest_data_dir.glob("spike_recorder-*.dat"))
if not all_nest_files:
    raise FileNotFoundError(f"No spike_recorder-*.dat files found in {nest_data_dir.resolve()}")

# Group files by recorder ID and rank, then load all recorders/layers.
# In your setup, recorder numbers 77170..77177 correspond to the 8 cortical layers,
# and rank 0..7 are thread-local shards that must be merged.
files_by_recorder = {}
for path in all_nest_files:
    m = nest_pattern.match(path.name)
    if m is None:
        continue
    recorder_id = int(m.group(1))
    rank_id = int(m.group(2))
    files_by_recorder.setdefault(recorder_id, {})[rank_id] = path

if not files_by_recorder:
    raise RuntimeError("Could not parse any spike_recorder files.")

expected_recorders = list(range(77170, 77178))
expected_ranks = set(range(8))
missing_recorders = [rid for rid in expected_recorders if rid not in files_by_recorder]
if missing_recorders:
    raise ValueError(f"Missing recorder IDs: {missing_recorders}")

# Map recorder IDs to layers in ascending order
recorder_to_layer = {rid: layer_labels[i] for i, rid in enumerate(expected_recorders)}

print("Using recorder IDs:", expected_recorders)
print("Recorder-to-layer mapping:", recorder_to_layer)

nest_parts = []
for recorder_id in expected_recorders:
    rank_map = files_by_recorder[recorder_id]
    missing_ranks = sorted(expected_ranks.difference(rank_map.keys()))
    if missing_ranks:
        raise ValueError(f"Recorder {recorder_id} is missing rank files: {missing_ranks}")

    layer_name = recorder_to_layer[recorder_id]
    layer_idx = layer_labels.index(layer_name)
    start_id = int(starts[layer_idx])
    end_id = int(ends[layer_idx])

    for rank_id in sorted(rank_map):
        fpath = rank_map[rank_id]
        part = pd.read_csv(fpath, sep=r"\s+", comment="#", engine="python")
        part.columns = [c.strip() for c in part.columns]
        if not {"sender", "time_ms"}.issubset(set(part.columns)):
            raise ValueError(f"Unexpected columns in {fpath.name}: {part.columns.tolist()}")

        part = part[["sender", "time_ms"]].rename(columns={"sender": "neuron_id", "time_ms": "spike_time_ms"})
        part["neuron_id"] = part["neuron_id"].astype(np.int64)
        part["spike_time_ms"] = part["spike_time_ms"].astype(float)
        part["recorder_id"] = recorder_id
        part["rank_id"] = rank_id
        part["layer"] = layer_name
        part["layer_idx"] = layer_idx

        # Validate recorder-layer consistency by neuron ID range
        bad = (part["neuron_id"] < start_id) | (part["neuron_id"] > end_id)
        if bad.any():
            n_bad = int(bad.sum())
            raise ValueError(
                f"Recorder {recorder_id} ({layer_name}) has {n_bad} spikes outside expected neuron range [{start_id}, {end_id}]"
            )

        nest_parts.append(part)

nest_spike_events = pd.concat(nest_parts, ignore_index=True)
nest_spike_events["layer"] = pd.Categorical(
    nest_spike_events["layer"],
    categories=layer_labels,
    ordered=True,
)
nest_spike_events = nest_spike_events.sort_values(["layer", "neuron_id", "spike_time_ms"]).reset_index(drop=True)

print(f"Loaded NEST events: {len(nest_spike_events):,}")
print("NEST events per layer:")
display(nest_spike_events.groupby("layer", observed=False).size().rename("n_events"))


def compute_layer_metrics(spike_events_df, layer_table_df, labels, t_stop_ms_value, max_corr_neurons=200, seed=42):
    rng_local = np.random.default_rng(seed)
    corr_bin_size_local = 2 * pq.ms

    by_neuron = {
        int(nid): grp["spike_time_ms"].to_numpy(dtype=float)
        for nid, grp in spike_events_df.groupby("neuron_id")
    }

    metric_dist = {}
    rows = []

    for row in layer_table_df.itertuples(index=False):
        layer_name = row.layer
        n_start = int(row.start_id)
        n_end = int(row.end_id)
        neuron_range = np.arange(n_start, n_end + 1, dtype=np.int64)

        rate_vals = []
        cv_vals = []
        spiking_trains = []

        for neuron_id in neuron_range:
            times_ms = by_neuron.get(int(neuron_id), np.empty(0, dtype=float))
            st = neo.SpikeTrain(times_ms * pq.ms, t_start=0.0 * pq.ms, t_stop=t_stop_ms_value * pq.ms)

            rate_vals.append(float(mean_firing_rate(st).rescale(pq.Hz).magnitude))

            if times_ms.size >= 2:
                cv_vals.append(float(cv(isi(st))))

            if times_ms.size > 0:
                spiking_trains.append(st)

        corr_offdiag = np.array([], dtype=float)
        if len(spiking_trains) >= 2:
            sample_n = min(max_corr_neurons, len(spiking_trains))
            sampled_idx = rng_local.choice(len(spiking_trains), size=sample_n, replace=False)
            sampled = [spiking_trains[i] for i in sampled_idx]

            binned = BinnedSpikeTrain(
                sampled,
                bin_size=corr_bin_size_local,
                t_start=0.0 * pq.ms,
                t_stop=t_stop_ms_value * pq.ms,
            )
            cm = np.asarray(corrcoef(binned, binary=False), dtype=float)
            corr_offdiag = cm[np.triu_indices_from(cm, k=1)]

        rate_vals = np.asarray(rate_vals, dtype=float)
        cv_vals = np.asarray(cv_vals, dtype=float)

        metric_dist[layer_name] = {
            "rate_hz": rate_vals,
            "cv_isi": cv_vals,
            "pearson_offdiag": corr_offdiag,
        }

        rows.append(
            {
                "layer": layer_name,
                "n_neurons_total": int(neuron_range.size),
                "n_spiking_neurons": int(len(spiking_trains)),
                "rate_mean_hz": float(np.nanmean(rate_vals)) if rate_vals.size else np.nan,
                "cv_isi_mean": float(np.nanmean(cv_vals)) if cv_vals.size else np.nan,
                "pearson_r_mean": float(np.nanmean(corr_offdiag)) if corr_offdiag.size else np.nan,
            }
        )

    summary = pd.DataFrame(rows)
    summary["layer"] = pd.Categorical(summary["layer"], categories=labels, ordered=True)
    summary = summary.sort_values("layer").reset_index(drop=True)
    return summary, metric_dist


# Use a common comparison window so rates/correlation are directly comparable
neuroring_max_ms = float(spike_events["spike_time_ms"].max()) if len(spike_events) else 0.0
nest_max_ms = float(nest_spike_events["spike_time_ms"].max()) if len(nest_spike_events) else 0.0
common_t_stop_ms = max(0.1, min(neuroring_max_ms, nest_max_ms) + 0.1)

spike_events_cmp = spike_events[spike_events["spike_time_ms"] <= common_t_stop_ms].copy()
nest_spike_events_cmp = nest_spike_events[nest_spike_events["spike_time_ms"] <= common_t_stop_ms].copy()

summary_nr_cmp, dist_nr_cmp = compute_layer_metrics(
    spike_events_cmp,
    layer_table,
    layer_labels,
    t_stop_ms_value=common_t_stop_ms,
    max_corr_neurons=200,
    seed=42,
)
summary_nest_cmp, dist_nest_cmp = compute_layer_metrics(
    nest_spike_events_cmp,
    layer_table,
    layer_labels,
    t_stop_ms_value=common_t_stop_ms,
    max_corr_neurons=200,
    seed=42,
)

comparison_summary = summary_nr_cmp.merge(
    summary_nest_cmp,
    on="layer",
    suffixes=("_neuroring", "_nest"),
)

print(f"Comparison window t_stop: {common_t_stop_ms:.3f} ms")
print(f"NeuroRing events used: {len(spike_events_cmp):,}")
print(f"NEST events used: {len(nest_spike_events_cmp):,}")
display(comparison_summary.round(4))

Using recorder IDs: [77170, 77171, 77172, 77173, 77174, 77175, 77176, 77177]
Recorder-to-layer mapping: {77170: '2/3E', 77171: '2/3I', 77172: '4E', 77173: '4I', 77174: '5E', 77175: '5I', 77176: '6E', 77177: '6I'}
Loaded NEST events: 2,445,370
NEST events per layer:


layer
2/3E    194104
2/3I    172417
4E      911102
4I      312087
5E      381390
5I       89954
6E      159120
6I      225196
Name: n_events, dtype: int64

/home/miahafiz/.local/lib/python3.10/site-packages/elephant/conversion.py:1130: UserWarning:Binning discarded 1 last spike(s) of the input spiketrain


Comparison window t_stop: 10000.000 ms
NeuroRing events used: 2,457,948
NEST events used: 2,445,370


,layer,n_neurons_total_neuroring,n_spiking_neurons_neuroring,rate_mean_hz_neuroring,cv_isi_mean_neuroring,pearson_r_mean_neuroring,n_neurons_total_nest,n_spiking_neurons_nest,rate_mean_hz_nest,cv_isi_mean_nest,pearson_r_mean_nest
0,2/3E,20683,19130,0.9270,0.6929,0.0040,20683,19218,0.9385,0.6970,0.0036
1,2/3I,5834,5813,2.9694,0.8275,0.0037,5834,5810,2.9554,0.8306,0.0035
2,4E,21915,21845,4.1732,0.8178,0.0041,21915,21832,4.1574,0.8176,0.0038
3,4I,5479,5470,5.7108,0.8226,0.0026,5479,5473,5.6961,0.8173,0.0024
4,5E,4850,4845,7.9996,0.7837,0.0087,4850,4844,7.8637,0.7805,0.0090
5,5I,1065,1065,8.4997,0.7549,0.0020,1065,1061,8.4464,0.7551,0.0019
6,6E,14395,12472,1.1209,0.6849,0.0009,14395,12366,1.1054,0.6832,0.0011
7,6I,2948,2945,7.6552,0.7541,0.0014,2948,2943,7.6389,0.7528,0.0010


In [9]:
# Single manuscript-style comparison figure: all layers x all statistics
# Layout: 4 rows (L23, L4, L5, L6) x 6 columns ([E,I] for each metric block)

layers_grid = [
    ("2/3E", "2/3I"),
    ("4E", "4I"),
    ("5E", "5I"),
    ("6E", "6I"),
]
metric_blocks = [
    ("rate_hz", "Firing Rate [Hz]"),
    ("cv_isi", "Coefficient of Variation of Inter-Spike Intervals"),
    ("pearson_offdiag", "Pearson Correlation"),
]

color_nest = "#3d5aa9"
color_nr = "#c94c4c"


def short_layer_name(layer_name):
    return "L" + layer_name.replace("/", "")


def clean_vec(x):
    x = np.asarray(x, dtype=float)
    return x[np.isfinite(x)]


def metric_xlim(metric_key, q_low=0.5, q_high=99.5):
    vals = []
    for (le, li) in layers_grid:
        vals.extend([clean_vec(dist_nest_cmp[le][metric_key]), clean_vec(dist_nest_cmp[li][metric_key])])
        vals.extend([clean_vec(dist_nr_cmp[le][metric_key]), clean_vec(dist_nr_cmp[li][metric_key])])
    vals = np.concatenate([v for v in vals if v.size > 0]) if any(v.size > 0 for v in vals) else np.array([0.0, 1.0])
    lo = float(np.percentile(vals, q_low))
    hi = float(np.percentile(vals, q_high))
    if metric_key in {"rate_hz", "cv_isi"}:
        lo = max(0.0, lo)
    if np.isclose(lo, hi):
        hi = lo + 1.0
    pad = 0.05 * (hi - lo)
    return lo - pad, hi + pad


xlims = {k: metric_xlim(k) for k, _ in metric_blocks}

#fig, axes = plt.subplots(nrows=4, ncols=6, figsize=(16, 9), constrained_layout=True)
fig, axes = plt.subplots(nrows=4, ncols=6, figsize=(14, 8), constrained_layout=True)

# Track y-max per metric so firing/CV/Pearson each use an appropriate density scale
metric_ymax = {k: 0.0 for k, _ in metric_blocks}
metric_axes = {k: [] for k, _ in metric_blocks}

for r, (layer_e, layer_i) in enumerate(layers_grid):
    layer_pair = [layer_e, layer_i]

    for mb, (metric_key, _) in enumerate(metric_blocks):
        for c_local, layer_name in enumerate(layer_pair):
            c = mb * 2 + c_local
            ax = axes[r, c]
            metric_axes[metric_key].append(ax)

            nest_vals = clean_vec(dist_nest_cmp[layer_name][metric_key])
            nr_vals = clean_vec(dist_nr_cmp[layer_name][metric_key])

            if nest_vals.size >= 2 and not np.allclose(nest_vals, nest_vals[0]):
                sns.kdeplot(nest_vals, ax=ax, color=color_nest, linewidth=1.5, fill=False, label="NEST")
                metric_ymax[metric_key] = max(metric_ymax[metric_key], float(np.nanmax(ax.lines[-1].get_ydata())))
            elif nest_vals.size > 0:
                ax.axvline(float(nest_vals.mean()), color=color_nest, linewidth=1.5, label="NEST")

            if nr_vals.size >= 2 and not np.allclose(nr_vals, nr_vals[0]):
                sns.kdeplot(nr_vals, ax=ax, color=color_nr, linewidth=1.3, linestyle="--", fill=False, label="NeuroRing")
                metric_ymax[metric_key] = max(metric_ymax[metric_key], float(np.nanmax(ax.lines[-1].get_ydata())))
            elif nr_vals.size > 0:
                ax.axvline(float(nr_vals.mean()), color=color_nr, linewidth=1.3, linestyle="--", label="NeuroRing")

            ax.set_xlim(*xlims[metric_key])
            ax.grid(True, alpha=0.25)

            # Layer indicator inside subplot
            ax.text(0.96, 0.90, short_layer_name(layer_name), transform=ax.transAxes,
                    ha="right", va="top", fontsize=10, fontweight="bold")

            # Only show legend once
            if r == 0 and c == 0:
                ax.legend(loc="center", frameon=True, fontsize=9)

            # Reduce visual clutter
            if r < 3:
                ax.set_xlabel("")
            if c > 0:
                ax.set_ylabel("")

    # y-axis label per row (probability density)
    axes[r, 0].set_ylabel("p")

# Apply one y-scale per metric block across all layers
for metric_key, _ in metric_blocks:
    ymax = metric_ymax[metric_key]
    if not np.isfinite(ymax) or ymax <= 0:
        ymax = 1.0
    for ax in metric_axes[metric_key]:
        ax.set_ylim(0, ymax * 1.08)

# Group x-labels for the three metric blocks (moved slightly lower to avoid overlap)
fig.text(0.17, -0.02, metric_blocks[0][1], ha="center", fontsize=14)
fig.text(0.50, -0.02, metric_blocks[1][1], ha="center", fontsize=14)
fig.text(0.83, -0.02, metric_blocks[2][1], ha="center", fontsize=14)

#fig.suptitle("Cortical Microcircuit Layer-wise Statistics: NEST vs NeuroRing", fontsize=16, fontweight="bold")

combined_png = "layerwise_all_stats_nest_vs_neuroring.png"
combined_pdf = "layerwise_all_stats_nest_vs_neuroring.pdf"
fig.savefig(combined_png, dpi=300, bbox_inches="tight")
fig.savefig(combined_pdf, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved combined figure: {combined_png}")
print(f"Saved combined figure: {combined_pdf}")

Saved combined figure: layerwise_all_stats_nest_vs_neuroring.png
Saved combined figure: layerwise_all_stats_nest_vs_neuroring.pdf


In [10]:
# Raster plot comparison (NEST vs NeuroRing) with configurable time range
# Requires: spike_events, nest_spike_events, layer_table, layer_labels

# -----------------------------
# User controls
# -----------------------------
# Time window for raster display (seconds)
time_start_s = 1.0
time_end_s = 1.4

# Marker style
marker_size = 1
alpha_val = 0.85

# -----------------------------
# Prepare boundaries and colors
# -----------------------------
time_start_ms = time_start_s * 1000.0
time_end_ms = time_end_s * 1000.0
if time_end_ms <= time_start_ms:
    raise ValueError("time_end_s must be larger than time_start_s")

layer_colors = {
    "2/3E": "#4C72B0",
    "2/3I": "#DD8452",
    "4E": "#4C72B0",
    "4I": "#DD8452",
    "5E": "#4C72B0",
    "5I": "#DD8452",
    "6E": "#4C72B0",
    "6I": "#DD8452",
}

layer_bounds = []
for row in layer_table.itertuples(index=False):
    layer_bounds.append((row.layer, int(row.start_id), int(row.end_id)))


def layer_centers_ticks(bounds):
    ticks = []
    labels = []
    for lname, start_id, end_id in bounds:
        ticks.append(0.5 * (start_id + end_id))
        labels.append(lname)
    return ticks, labels


def add_layer_guides(ax, bounds):
    # Horizontal separators between layers
    for _, _, end_id in bounds[:-1]:
        ax.axhline(end_id + 0.5, color="gray", linewidth=0.6, alpha=0.5)


# -----------------------------
# Filter events by time window
# -----------------------------
nr_win = spike_events[(spike_events["spike_time_ms"] >= time_start_ms) & (spike_events["spike_time_ms"] <= time_end_ms)].copy()
nest_win = nest_spike_events[(nest_spike_events["spike_time_ms"] >= time_start_ms) & (nest_spike_events["spike_time_ms"] <= time_end_ms)].copy()

# Convert x-axis to seconds for plotting
nr_win["time_s"] = nr_win["spike_time_ms"] / 1000.0
nest_win["time_s"] = nest_win["spike_time_ms"] / 1000.0

# -----------------------------
# Plot side-by-side raster
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True, constrained_layout=True)
plot_data = [(axes[0], nest_win, "NEST"), (axes[1], nr_win, "NeuroRing")]

for ax, df_plot, title in plot_data:
    for lname, start_id, end_id in layer_bounds:
        sub = df_plot[(df_plot["neuron_id"] >= start_id) & (df_plot["neuron_id"] <= end_id)]
        if len(sub) == 0:
            continue
        ax.scatter(
            sub["time_s"].to_numpy(dtype=float),
            sub["neuron_id"].to_numpy(dtype=float),
            s=marker_size,
            c=layer_colors.get(lname, "#4C72B0"),
            marker=".",
            alpha=alpha_val,
            linewidths=0,
            rasterized=True,
        )

    add_layer_guides(ax, layer_bounds)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("time (s)")
    ax.set_xlim(time_start_s, time_end_s)
    ax.grid(True, axis="x", alpha=0.25)

axes[0].set_ylabel("neuron id")

# Layer labels at midpoints
yticks, ylabels = layer_centers_ticks(layer_bounds)
axes[0].set_yticks(yticks)
axes[0].set_yticklabels(ylabels)

# Keep identical y-range across both panels
global_min = int(layer_bounds[0][1])
global_max = int(layer_bounds[-1][2])
axes[0].set_ylim(global_max + 1, global_min - 1)  # invert so 2/3 is at top, like paper style

#fig.suptitle("Layer-wise raster comparison", fontsize=15, fontweight="bold")

out_png = "raster_nest_vs_neuroring.png"
out_pdf = "raster_nest_vs_neuroring.pdf"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: {out_png}")
print(f"Saved: {out_pdf}")
print(f"Window: {time_start_s:.3f}s to {time_end_s:.3f}s")

Saved: raster_nest_vs_neuroring.png
Saved: raster_nest_vs_neuroring.pdf
Window: 1.000s to 1.400s
